In [8]:

from roboflow import Roboflow
rf = Roboflow(api_key="O9LQp0Izd42qdBKfBuae")
project = rf.workspace("ritm").project("infaprim-clf-sdrmg")
version = project.version(7)
dataset = version.download("folder")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to infaprim-clf-7 in folder:: 100%|██████████| 44548/44548 [00:21<00:00, 2043.17it/s]


In [2]:
! pip install -q timm
! pip install -q pytorch-lightning
! pip install split-folders
! pip install ipywidgets

In [9]:
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
import random
from tqdm import tqdm
import torch
from torchvision import transforms
from pathlib import Path
from glob import glob

contrast_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)


class ProductDataset(Dataset):
    def __init__(self, cls_folder):
        classes_images = []
        labels = []

        classes = sorted(os.listdir(cls_folder))
        for class_name in classes:
            class_images = []
            class_path = os.path.join(cls_folder, class_name)

            for img_name in tqdm(os.listdir(class_path), desc=f"Loading {class_name}"):
                if img_name.startswith("."):
                  continue
                img_path = os.path.join(class_path, img_name)
                class_images.append(img_path)

            # if len(class_images) < 400:
            #     aug_images = Path("data/concurrent/aug_images")
            #     class_aug_images = aug_images / class_name
            #     aug_class_images = glob(str(class_aug_images / "*" / "*"))
            #     class_images.extend(aug_class_images)

            classes_images.append(class_images)
            labels.append(class_name)

        # others_images = []
        # for img_name in os.listdir(others_folder):
        #     img_path = os.path.join(others_folder, img_name)
        #     others_images.append(img_path)

        # for class_name in os.listdir("data/concurrent/aug_images"):
        #     if class_name in roi_classes:
        #         continue
        #     class_images = glob(os.path.join("data/concurrent/aug_images",class_name,"*","*"))
        #     others_images.extend(class_images)

        # classes_images.append(others_images)
        # labels.append("others")

        self.labels = labels
        self.classes_images = classes_images

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        images_1 = []
        images_2 = []
        labels = []

        for i, label in enumerate(self.labels):
            labels.append(label)

            rand_images = random.choices(self.classes_images[i], k=2)

            for j, img_path in enumerate(rand_images):
                img = Image.open(img_path)
                img_enc = contrast_transforms(img)

                if j % 2:
                    images_2.append(img_enc)
                else:
                    images_1.append(img_enc)
            #
            # img_2 = Image.open(img_2_path)
            # img_enc_2 = processor(img_2, return_tensors="pt")['pixel_values'][0]

            # images_1.append(img_enc_1)
            # images_2.append(img_enc_2)

        images_1.insert(0, images_1.pop(idx))
        images_2.insert(0, images_2.pop(idx))

        images_1 = torch.stack(images_1).squeeze()
        images_2 = torch.stack(images_2).squeeze()

        return [images_1, images_2]

In [10]:
ds = ProductDataset(
    cls_folder=r"C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\train"
    )

dl = DataLoader(dataset=ds, batch_size=8)

Loading v_rice_corn_banana-3: 100%|██████████| 124/124 [00:00<?, ?it/s]


In [11]:
next(iter(dl))

[tensor([[[[[0.5686, 0.5686, 0.5725,  ..., 0.3255, 0.3176, 0.3137],
            [0.5647, 0.5647, 0.5686,  ..., 0.3412, 0.3216, 0.3098],
            [0.5608, 0.5608, 0.5647,  ..., 0.3529, 0.3255, 0.3059],
            ...,
            [0.3725, 0.3608, 0.3451,  ..., 0.7294, 0.7333, 0.7373],
            [0.3686, 0.3608, 0.3529,  ..., 0.7333, 0.7529, 0.7686],
            [0.3608, 0.3608, 0.3608,  ..., 0.7451, 0.7765, 0.8000]],
 
           [[0.5412, 0.5412, 0.5451,  ..., 0.2980, 0.2902, 0.2863],
            [0.5373, 0.5373, 0.5412,  ..., 0.3137, 0.2941, 0.2824],
            [0.5373, 0.5373, 0.5373,  ..., 0.3255, 0.2980, 0.2784],
            ...,
            [0.3765, 0.3647, 0.3490,  ..., 0.7569, 0.7608, 0.7647],
            [0.3725, 0.3647, 0.3569,  ..., 0.7608, 0.7804, 0.7961],
            [0.3647, 0.3647, 0.3647,  ..., 0.7725, 0.8039, 0.8275]],
 
           [[0.4667, 0.4667, 0.4706,  ..., 0.5608, 0.5529, 0.5490],
            [0.4627, 0.4627, 0.4667,  ..., 0.5765, 0.5569, 0.5451],
        

In [13]:
val_ds = ProductDataset(
    cls_folder=r"C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\valid"
    )
val_dl = DataLoader(dataset=val_ds, batch_size=8)

Loading v_rice_corn_banana-3: 100%|██████████| 30/30 [00:00<?, ?it/s]


In [14]:
next(iter(val_dl))

[tensor([[[[[0.1137, 0.1137, 0.1098,  ..., 0.1373, 0.1373, 0.1373],
            [0.1255, 0.1255, 0.1255,  ..., 0.1333, 0.1333, 0.1333],
            [0.1412, 0.1412, 0.1529,  ..., 0.1216, 0.1255, 0.1255],
            ...,
            [0.1647, 0.1647, 0.2000,  ..., 0.2196, 0.2431, 0.2431],
            [0.1176, 0.1176, 0.1608,  ..., 0.2353, 0.2588, 0.2588],
            [0.0863, 0.0863, 0.1373,  ..., 0.2471, 0.2706, 0.2706]],
 
           [[0.1490, 0.1490, 0.1451,  ..., 0.1373, 0.1373, 0.1373],
            [0.1608, 0.1608, 0.1608,  ..., 0.1333, 0.1333, 0.1333],
            [0.1765, 0.1765, 0.1882,  ..., 0.1216, 0.1255, 0.1255],
            ...,
            [0.1804, 0.1804, 0.2157,  ..., 0.1961, 0.2196, 0.2196],
            [0.1333, 0.1333, 0.1765,  ..., 0.2118, 0.2353, 0.2353],
            [0.1020, 0.1020, 0.1529,  ..., 0.2235, 0.2471, 0.2471]],
 
           [[0.2706, 0.2706, 0.2667,  ..., 0.1765, 0.1765, 0.1765],
            [0.2863, 0.2863, 0.2863,  ..., 0.1725, 0.1765, 0.1765],
        

In [15]:
import timm

vit_model = timm.create_model("vit_tiny_patch16_224", pretrained=True)
vit_model.fc_norm = torch.nn.Identity()
vit_model.head_drop = torch.nn.Identity()
vit_model.head = torch.nn.Identity()

vit_model

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)


In [16]:
import pytorch_lightning as pl
from torch import nn
from torch import optim
import torch.nn.functional as F

class SimCLR(pl.LightningModule):
    def __init__(self, hidden_dim, lr, temperature, weight_decay, max_epochs=500):
        super().__init__()

        self.save_hyperparameters()

        assert self.hparams.temperature > 0.0, "The temperature must be a positive float!"

        self.convnet = timm.create_model("vit_tiny_patch16_224", pretrained=True).train()

        self.convnet.fc_norm = torch.nn.Identity()
        self.convnet.head_drop = torch.nn.Identity()
        self.convnet.head = torch.nn.Identity()

        self.convnet.fc_norm.requires_grad_(requires_grad=False)
        self.convnet.head_drop.requires_grad_(requires_grad=False)
        self.convnet.head.requires_grad_(requires_grad=False)

        self.projection = nn.Sequential(
            nn.Linear(192, 192),
            nn.ReLU(inplace=True),
            nn.Linear(192, 192),
        )

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams.max_epochs,
                                                            eta_min=self.hparams.lr / 50)
        return [optimizer], [lr_scheduler]

    def info_nce_loss(self, batch, mode="train"):
        images = batch
        images = torch.cat([images[0].squeeze(), images[1].squeeze()], dim=0)

        # Encode images
        features = self.convnet(images)
        features = self.projection(features)

        # Calculate Cosine Similarity
        cos_sim = F.cosine_similarity(features[:, None, :], features[None, :, :], dim=-1)

        # Mask out cosine similarity to itself
        self_mask = torch.eye(cos_sim.shape[0], dtype=torch.bool, device=cos_sim.device)
        cos_sim.masked_fill_(self_mask, -9e15)

        # Find positive example -> batch_size//2 away from the original example
        pos_mask = self_mask.roll(shifts=cos_sim.shape[0] // 2, dims=0)

        # InfoNCE loss
        cos_sim = cos_sim / self.hparams.temperature
        nll = -cos_sim[pos_mask] + torch.logsumexp(cos_sim, dim=-1)
        nll = nll.mean()

        # Logging loss
        self.log(mode + "_loss", nll, prog_bar=True)
        # Get ranking position of positive example
        comb_sim = torch.cat(
            [cos_sim[pos_mask][:, None], cos_sim.masked_fill(pos_mask, -9e15)],  # First position positive example
            dim=-1,
        )

        sim_argsort = comb_sim.argsort(dim=-1, descending=True).argmin(dim=-1)
        # Logging ranking metrics
        self.log(mode + "_acc_top1", (sim_argsort == 0).float().mean(), prog_bar=True)
        self.log(mode + "_acc_top5", (sim_argsort < 5).float().mean(), prog_bar=True)
        self.log(mode + "_acc_mean_pos", 1 + sim_argsort.float().mean(), prog_bar=True)

        return nll

    def training_step(self, batch, batch_idx):
        return self.info_nce_loss(batch, mode="train")


model = SimCLR(hidden_dim=128, lr=5e-4, temperature=0.07, weight_decay=1e-4, max_epochs=500)

W1026 15:57:35.762000 27316 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [17]:
print("Dataset length:", len(dl.dataset))
print("Batch count:", len(dl))
print("Batch size:", dl.batch_size)
print("Dataset class:", type(dl.dataset))

Dataset length: 1
Batch count: 1
Batch size: 8
Dataset class: <class '__main__.ProductDataset'>


In [18]:
trainer = pl.Trainer(
    accelerator="auto",
    devices=1,
    max_epochs=1000,
    accumulate_grad_batches=8
)

trainer.fit(model=model, train_dataloaders=dl)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, 

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ convnet    │ VisionTransformer │  5.5 M │ train │     0 │
│ 1 │ projection │ Sequential        │ 74.1 K │ train │     0 │
└───┴────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 5.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.6 M                                                                                                
Total estimated model params size (MB): 22.394                                                                     
Modules in train mode: 280                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:321: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


`Trainer.fit` stopped: `max_epochs=1000` reached.


In [19]:
torch.save(model.convnet, "SimCLR_VIT_1000epchs.pth")

In [ ]:
! pip install plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 2.4 MB/s eta 0:00:04
   --- ------------------------------------ 0.8/9.7 MB 2.4 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/9.7 MB 2.3 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.7 MB 1.7 MB/s eta 0:00:05
   ------ --------------------------------- 1.6/9.7 MB 1.4 MB/s eta 0:00:06
   -------- ------------------------------- 2.1/9.7 MB 1.6 MB/s eta 0:00:05
   ------------ --------------------------- 3.1/9.7 MB 2.1 MB/s eta 0:00:04
   --------------- ------------------------ 3.7/9.7 MB 2.2 MB/s eta 0:00:03
   ----------------- ---------------------- 4.2/9.7 MB 2.3 MB/s eta 0:00:03
   ------------------- -------------------- 4.7/9.7 MB 2.3 MB/s eta 0:00:03
   -------------------- ------------------- 5.0/9.7 MB 2.2 MB/s eta 0:00:03
   --------------------- -

In [12]:
! pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 441.3 kB/s eta 0:00:18
   -- ------------------------------------- 0.5/8.3 MB 441.3 kB/s eta 0:00:18
   --- ------------------------------------ 0.8/8.3 MB 486.4 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/8.3 MB 486.4 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/8.3 MB 486.4 kB/s eta 0:00:16
   --- ------------------------------------ 0.8/8.3 MB 486

In [1]:
! pip install --upgrade plotly ipython nbformat

In [21]:
import numpy as np
from sklearn.decomposition import PCA
import plotly.express as px

import plotly.io as pio
pio.renderers.default = 'colab'


vit_model = torch.load(r"C:\Users\Administrator\Desktop\Coding\SimCLR_VIT_1000epchs.pth", weights_only=False)


def visualize(images, labels, encoder):
    embeddings = []
    classes = []
    for class_images, label in zip(images, labels):
        class_images = random.choices(class_images, k=30)
        class_images = list(map(Image.open, class_images))
        class_images = list(map(contrast_transforms, class_images))
        class_images = torch.stack(class_images)

        with torch.no_grad():
            class_embeddings = encoder(class_images).cpu().numpy()
            print(class_embeddings.shape)
        embeddings.append(class_embeddings)
        classes.extend([label] * len(class_images))

    pca = PCA(n_components=2)
    embeddings_np = np.vstack(embeddings)
    embeddings_pca = pca.fit_transform(embeddings_np)

    # fig = px.scatter_3d(x=embeddings_pca[:, 0], y=embeddings_pca[:, 1], z=embeddings_pca[:, 2], color=classes)
    fig = px.scatter(x=embeddings_pca[:, 0], y=embeddings_pca[:, 1], color=classes)
    return fig

visualize(val_ds.classes_images, ds.labels, vit_model).show()

(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)


In [22]:
visualize(val_ds.classes_images, ds.labels, vit_model)

(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)
(30, 192)


In [23]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

contrast_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)


class ClfDS(Dataset):
    def __init__(self, folder):
        self.folder = folder
        classes=[]
        images=[]
        for cls in os.listdir(self.folder):
          for files in os.listdir(os.path.join(self.folder, cls)):
            img_path =os.path.join(self.folder, cls, files)
            images.append(img_path)
            classes.append(cls)
        self.id2label = {idx: label for idx, label in enumerate(sorted(set(classes)))}
        self.label2id = {label: idx for idx, label in enumerate(sorted(set(classes)))}
        self.classes = classes
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        id=self.classes[idx]
        img = Image.open(img_path)
        img = contrast_transforms(img)

        return img, self.label2id[id]

In [49]:
from pathlib import Path

base = Path(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7')
train_cls = {p.name for p in (base / 'train').iterdir() if p.is_dir()}
val_cls   = {p.name for p in (base / 'valid').iterdir() if p.is_dir()}

print("train:", len(train_cls), "val:", len(val_cls))
print("faqat trainda:", sorted(train_cls - val_cls))
print("faqat valda:  ", sorted(val_cls - train_cls))

train: 173 val: 172
faqat trainda: ['v_pumpkin']
faqat valda:   []


In [50]:
from torch.utils.data import DataLoader
import os
train_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\train'), batch_size=32, shuffle=True)
val_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\valid'), batch_size=32, shuffle=False)

In [51]:
next(iter(val_dl))

[tensor([[[[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [0.9451, 0.9529, 0.9647,  ..., 1.0000, 0.9961, 1.0000],
           [0.8980, 0.9020, 0.9098,  ..., 1.0000, 0.9961, 0.9961],
           ...,
           [0.9804, 0.9765, 0.9804,  ..., 1.0000, 1.0000, 1.0000],
           [0.8667, 0.8627, 0.8745,  ..., 1.0000, 1.0000, 1.0000],
           [0.6784, 0.6706, 0.6706,  ..., 1.0000, 1.0000, 1.0000]],
 
          [[0.6196, 0.6157, 0.6000,  ..., 0.3529, 0.3490, 0.3569],
           [0.4118, 0.4196, 0.4275,  ..., 0.3529, 0.3490, 0.3529],
           [0.3333, 0.3373, 0.3412,  ..., 0.3569, 0.3490, 0.3529],
           ...,
           [0.9569, 0.9490, 0.9373,  ..., 0.3137, 0.3137, 0.3137],
           [0.7804, 0.7765, 0.7843,  ..., 0.3137, 0.3137, 0.3137],
           [0.5686, 0.5608, 0.5647,  ..., 0.3137, 0.3137, 0.3137]],
 
          [[0.2941, 0.2902, 0.2784,  ..., 0.0078, 0.0078, 0.0196],
           [0.0902, 0.0980, 0.1059,  ..., 0.0078, 0.0000, 0.0118],
           [0.0157, 0.01

In [8]:
import timm
import torch

vit_model = timm.create_model("vit_tiny_patch16_224", pretrained=False)



In [61]:
len(set(val_dl.dataset.classes))

172

In [1]:
from torch import nn, optim
import torch
import torch.nn.functional as F


class ViTClf(nn.Module):
    def __init__(self):
        super().__init__()

        model = torch.load(r"vit_tiny_patch16_224", weights_only=False)
        model.requires_grad_(False)

        model.fc_norm = vit_model.fc_norm
        model.head_drop = vit_model.head_drop
        model.head = torch.nn.Linear(in_features=192, out_features=len(set(train_dl.dataset.classes)))
        #
        # model.fc_norm.requires_grad_(True)
        # model.head_drop.requires_grad_(True)
        # model.head.requires_grad_(True)

        self.clf = model

    def forward(self, images):
        clf_out = self.clf(images)
        model = torch.load(r"vit_tiny_patch16_224", weights_only=False)
        model.requires_grad_(False)

        model.fc_norm = vit_model.fc_norm
        model.head_drop = vit_model.head_drop
        model.head = torch.nn.Linear(in_features=192, out_features=len(set(train_dl.dataset.classes)))

        return clf_out


In [2]:
import pytorch_lightning as L


class LitAutoEncoder(L.LightningModule):
    def __init__(
            self,
    ):
        super().__init__()
        self.model = ViTClf().cuda()

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        images, labels = batch
        preds = self.model(images)

        loss = F.cross_entropy(preds, labels)

        self.log("Loss", loss.item(), prog_bar=True)

        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer


model = LitAutoEncoder()

W1027 13:15:01.578000 24140 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


FileNotFoundError: [Errno 2] No such file or directory: 'vit_tiny_patch16_224'

In [71]:
trainer = L.Trainer(max_epochs=10, check_val_every_n_epoch=1)
trainer.fit(model=model, train_dataloaders=train_dl, val_dataloaders=val_dl)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ViTClf │  5.6 M │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 33.2 K                                                                                           
Non-trainable params: 5.5 M                                                                                        
Total params: 5.6 M                                                                                                
Total estimated model params size (MB): 22.230                                                                     
Modules in train mode: 277                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


`Trainer.fit` stopped: `max_epochs=10` reached.


In [72]:
torch.save(model.model, "VIT_clf.pth")

In [73]:
model=torch.load(r"C:\Users\Administrator\Desktop\Coding\VIT_clf.pth", weights_only=False)
model.eval()

ViTClf(
  (clf): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (norm): Identity()
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=192, out_features=768, bias=True)
          (act): GELU(

In [74]:
clf_model = model.cuda()
import torch
import torch.nn.functional as F

from sklearn.metrics import classification_report
import numpy as np

all_preds = []
all_labels = []

for batch in val_dl:
    images, labels = batch
    val_out = clf_model(images.cuda())

    probs = F.softmax(val_out, dim=-1)
    preds = torch.argmax(probs, dim=-1).cpu().numpy()
    labels = labels.cpu().numpy()

    preds = [val_dl.dataset.id2label[idx] for idx in preds]
    labels = [val_dl.dataset.id2label[idx] for idx in labels]

    all_preds.extend(preds)
    all_labels.extend(labels)

print(classification_report(all_labels, all_preds))

                          precision    recall  f1-score   support

                n_1_1050       0.83      0.88      0.86        17
              n_1_1050-2       0.65      0.63      0.64        27
              n_1_1050-3       0.89      0.94      0.91       139
                 n_1_300       0.83      0.95      0.89        79
               n_1_300-2       0.86      0.70      0.78        71
               n_1_300-3       0.91      0.92      0.91       182
                 n_1_600       0.77      0.93      0.84        40
               n_1_600-2       0.73      0.62      0.67        48
               n_1_600-3       0.93      0.88      0.90       163
                n_2_1050       0.89      0.89      0.89         9
              n_2_1050-2       0.65      0.69      0.67        32
              n_2_1050-3       0.91      0.87      0.89       106
                 n_2_300       0.81      0.95      0.88        81
               n_2_300-2       0.75      0.68      0.71        65
         

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
